# IMPORTS

In [ ]:
# Import future functions and system tools
from __future__ import print_function
import sys
import os
import argparse
import random
import gc
import math
import time
import logging
import abc
import csv
import pickle

# Import data handling and manipulation libraries
import pandas as pd
import numpy as np
from datetime import datetime
from itertools import chain
from multiprocessing import Pool
from typing import List, Tuple, Union

# Import GPU utilities and system monitoring
# import GPUtil

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from pylab import rcParams
import plotly.graph_objects as go
import plotly.offline as pyo
from IPython.display import HTML
%matplotlib inline

# Machine learning and preprocessing tools
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.preprocessing import MinMaxScaler,StandardScaler, RobustScaler
from sklearn.decomposition import PCA, FastICA
from statsmodels.robust.scale import mad

# PyTorch imports for deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.backends import cudnn
from torch.autograd import Variable
from torch.nn import TransformerEncoder, TransformerDecoder, TransformerEncoderLayer, TransformerDecoderLayer

# Statistical libraries
from scipy.stats import multivariate_normal, lognorm, norm, chi

# Progress bars and visual feedback
from tqdm import tqdm, trange, notebook

# Environment configuration to avoid kernel issues
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# Set up matplotlib and seaborn visual styles
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
HAPPY_COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(HAPPY_COLORS_PALETTE))
rcParams['figure.figsize'] = 18, 5

# Define device for PyTorch computations
ngpu = 1
device = torch.device("cuda" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Display GPU info
!nvidia-smi -L

# Clear memory and set random seed for reproducibility
gc.collect()
torch.cuda.empty_cache()
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print(DEVICE)


from data_utils import *
from train_utils import *
from Model_MTSAD import *



# INITIALIZE

In [ ]:
# Dataset Configuration
USED_DATASET = 'swat'  # Options: 'swat', 'wadi', 'smap', 'psm'
TRAIN_AMOUNT =0.3  # Proportion of data to use for training
VAL_AMOUNT = 1.0 - TRAIN_AMOUNT
anomaly_criteria = 'one'
# Model and Execution Settings
MODEL_NAME = 'MTSAD'  # Model options: 'MTSAD','MTSADver2', 'MTSAD_AttGAN', 'MTSAD_GAN', 'MTSAD_CATTGAN'
NEED_VALID = True  # Flag to indicate the requirement of validation set

# Noise Configuration
NEED_NOISE = True
NOISE_STAT = 'WithNoise'  # Options: 'WithNoise', 'WithoutNoise'

# Training Parameters
NUM_EPOCHS = 3
BATCH_SIZE = 64
SEQ_LEN = 30
SHIFT_LEN = 1
OPTIMIZER = 'Adam'
CRITERION = 'MSE' #Options are based on code
LEARNING_RATE = 0.001
HIDDEN = 128  # Hidden dimension
NORMALIZATION = 'MinMax' 
# Feature Dimensions Based on Dataset
FEATURES_BY_DATASET = {'swat': 51, 'wadi': 123}
NB_FEATURE = FEATURES_BY_DATASET.get(USED_DATASET, None)
LABEL_CRIT='one' # Options: 'one', 'half'

# DATASET

## LOAD DATASET

In [ ]:

print(f'STARTING READ AND PROCESS DATASET <{USED_DATASET}>')
if NEED_VALID:
    normal, val, attacked, mean, std, input_dim = load_dataset(
        USED_DATASET, TRAIN_AMOUNT, NEED_NOISE, NEED_VALID)
else:
    normal, attacked, mean, std, input_dim = load_dataset(
        USED_DATASET, TRAIN_AMOUNT, NEED_NOISE, NEED_VALID)

In [ ]:
plt.plot(attacked)
plt.show()
attacked.columns
plt.plot(attacked['Normal/Attack'])

## PROCESS DATASET

In [ ]:
print(f'Creating Datasets: {USED_DATASET}....')
print('TRAIN Dataset')

# Initialize training dataset
train = MyDataset(normal, seq_length=SEQ_LEN, shift_length=SHIFT_LEN, FS='none', is_train=True, normalization=NORMALIZATION)

# Initialize validation dataset if needed
if NEED_VALID:
    valid = MyDataset(val, seq_length=SEQ_LEN, shift_length=SHIFT_LEN, FS='none', is_train=False, normalization=NORMALIZATION,
                  median=train.median, mad_values=train.mad_values, scaler=train.scaler if NORMALIZATION in ['MinMax', 'RobustScaler'] else None)

# Initialize test dataset
print('\nTEST Dataset')
test = MyDataset(attacked, seq_length=SEQ_LEN, shift_length=SHIFT_LEN, FS='none', is_train=False, normalization=NORMALIZATION,
             median=train.median, mad_values=train.mad_values, scaler=train.scaler if NORMALIZATION in ['MinMax', 'RobustScaler'] else None)


# Print dataset information for verification
print()
print('Dataset shapes and contents:')
print('Train shape:', np.shape(train.data))
if NEED_VALID:
    print('Validation shape:', np.shape(valid.data) if valid else 'No validation set')
print('Test shape:', np.shape(test.data))



## DATALOADER

In [ ]:
# Initialize DataLoaders for train, validation, and test datasets
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=False)

if NEED_VALID:
    val_loader = DataLoader(valid, batch_size=BATCH_SIZE, shuffle=False)

test_loader = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False)

# Print shape of test labels for verification
print('Test labels shape:', np.shape(test_loader.dataset.label))


## GENERATE LABLES

In [ ]:
target_list, testing_arr = generate_labels(test_loader, device, anomaly_criteria)

In [ ]:
target_list.count(1)

# MODEL INITIALIZATION

In [ ]:
model = MTSAD(feats=NB_FEATURE, hidden_dim=HIDDEN, seq_len=SEQ_LEN)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

model

# TEST MODEL

In [ ]:
#### print("a sample from the train_loader:")
for i, (data, label) in enumerate(train_loader):
    data=data.to(device)
    print(data[0])
    label=label.to(device)
    break;
model=model.to(device)


generated_x = model(data.to(device))
plt.plot(data[0].detach().cpu().numpy())
plt.title(f'REAL & Label = {label[0].tolist()}')
plt.show()
plt.plot(generated_x[0].detach().cpu().numpy())
plt.title(f'RECONSTRUCTED& Label = {label[0].tolist()}')
plt.show()

# TRAIN 

In [ ]:
total_training_time = 0.0
for epoch in range(NUM_EPOCHS):
    train_time, input_data, output_data = train_epochs(epoch, NUM_EPOCHS, model, optimizer, criterion, train_loader, device)
    total_training_time += train_time

    # Plot real and reconstructed signals
    plot_signals(input_data, output_data)

    # Evaluation Phase
    eval_time = evaluate_epoch(epoch, NUM_EPOCHS, model, test_loader, device, target_list)

    # Move model back to GPU after evaluation
    model = model.to(device)

    print(f'Train time = {train_time}\n')
    print(f'eval time = {eval_time}\n')

# Post-Train Plots

In [ ]:
# Plot training and test losses over epochs
plt.plot(losses_train, label='Train Loss')
plt.plot(losses_test, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Over Epochs')
plt.legend()
plt.show()

# Plot metrics over epochs
plt.plot(recall_hist, label='Recall')
plt.plot(f1_hist, label='F1')
plt.plot(precision_hist, label='Precision')
plt.plot(auc_hist, label='AUC')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Metrics Over Epochs')
plt.legend()
plt.show()

# Plot confusion matrix components over epochs
plt.plot(TP_hist, label='TP')
plt.plot(TN_hist, label='TN')
plt.plot(FP_hist, label='FP')
plt.plot(FN_hist, label='FN')
plt.xlabel('Epoch')
plt.ylabel('Count')
plt.title('Confusion Matrix Components Over Epochs')
plt.legend()
plt.show()

# Combined plot of confusion matrix components and metrics
plt.plot(TP_hist, label='TP')
plt.plot(TN_hist, label='TN')
plt.plot(FP_hist, label='FP')
plt.plot(FN_hist, label='FN')
plt.plot(recall_hist, label='Recall')
plt.plot(f1_hist, label='F1')
plt.plot(precision_hist, label='Precision')
plt.plot(auc_hist, label='AUC')
plt.xlabel('Epoch')
plt.ylabel('Score/Count')
plt.title('Metrics and Confusion Matrix Components Over Epochs')
plt.legend()
plt.show()

print(f'Total Training Time = {total_training_time}')